In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train = pd.read_csv("../datasets/food_drug_pairs_train.csv")

print("Dataset shape:", train.shape)
train.head()

Dataset shape: (7152, 24)


,drug_index,drug_name,drug_contains,combined_text,Chemical_Class,Habit_Forming,Therapeutic_Class,Action_Class,food_name,energy,...,calcium,iron,vitamin_c,vitamin_a,vitamin_k_proxy,is_alcohol,is_leafy_green,meal_type,severity_silver,reason_tags_silver
0,122,Atorva 40 Tablet,Atorvastatin (40mg),Atorva 40 Tablet is a widely prescribed medici...,716,1,11,190,Kimchi Jjigae (1 bowl),350.00000,...,0.000000,0.00000,0.000000,0.000000,0,0,0,Dinner,1,high_fat_empty_stomach
1,42,Aromasin 25mg Tablet,Exemestane (25mg),Aromasin 25mg Tablet is also used in combinati...,99,1,6,0,Lassi (Mango 1 cup),180.00000,...,0.000000,0.00000,0.000000,0.000000,0,0,0,Snack,0,NaN
2,133,Atorva Tablet,Atorvastatin (10mg),Atorva Tablet is a widely prescribed medicine ...,716,1,11,190,"Gravy (coconut milk, thin)",256.73489,...,20.416855,1.87353,4.015721,5.107943,0,0,0,0,1,high_fat_empty_stomach
3,97,Aziderm 20% Cream,Azelaic Acid (20% w/w),Aziderm 20% Cream is for external use only. Yo...,456,1,13,67,"CASSAVA ROOT, FRESH,BOILED",130.00000,...,17.000000,0.50000,17.000000,0.000000,0,0,0,0,0,NaN
4,95,Anafortan 25 mg/300 mg Tablet,Camylofin (25mg)+ Paracetamol (300mg),Anafortan 25 mg/300 mg Tablet is taken with or...,0,1,14,0,"ASH GOURD, BOILED, W/ COCONUT MILK",134.00000,...,15.000000,0.58000,4.300000,5.000000,0,0,0,0,0,NaN


In [2]:
print(train["severity_silver"].value_counts())
print("\nPercentage:")
print(train["severity_silver"].value_counts(normalize=True) * 100)

severity_silver
0    5000
1    2128
2      24
Name: count, dtype: int64

Percentage:
severity_silver
0    69.910515
1    29.753915
2     0.335570
Name: proportion, dtype: float64


In [3]:
features = [
    "Chemical_Class",
    "Habit_Forming",
    "Therapeutic_Class",
    "Action_Class",
    "energy",
    "protein",
    "fat",
    "carbs",
    "fiber",
    "calcium",
    "iron",
    "vitamin_c",
    "vitamin_a",
    "vitamin_k_proxy",
    "is_alcohol",
    "is_leafy_green",
    "meal_type"
]

target = "severity_silver"

df = train[features + [target]].copy()

# Missing values handle
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(0)

# Encode categorical columns
encoders = {}

categorical_cols = [
    "Chemical_Class",
    "Therapeutic_Class",
    "Action_Class",
    "meal_type"
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))

Accuracy: 0.9846261355695318

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1000
           1       0.97      0.98      0.98       426
           2       0.83      1.00      0.91         5

    accuracy                           0.98      1431
   macro avg       0.93      0.99      0.96      1431
weighted avg       0.98      0.98      0.98      1431


Confusion Matrix:
[[986  13   1]
 [  8 418   0]
 [  0   0   5]]


In [4]:
joblib.dump(model, "../models/food_drug_risk_model.pkl")
joblib.dump(encoders, "../models/encoders.pkl")

print("Model saved successfully")

Model saved successfully
